# Notebook de test — Boussole, l'agent IA de l'ISC Business School

Ce notebook teste l'agent **smolagents** et ses **6 tools** en dehors de l'interface web.

**Prérequis :**
1. `pip install -r requirements.txt`
2. `python manage.py migrate` puis `python manage.py peupler_demo`
3. Un fichier `.env` à la racine avec `HF_TOKEN=hf_...` (sections 3 et 4 uniquement — les tools de base de données (section 2) fonctionnent sans token)

Lancer Jupyter **depuis la racine du projet** : `jupyter notebook`.

## 1. Initialisation de Django

In [ ]:
import os
import django

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'ISCAGENTProject.settings')
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = 'true'  # ORM dans Jupyter
django.setup()

from django.conf import settings
print('Django prêt.')
print('Token HF configuré :', 'oui' if settings.HF_TOKEN else 'NON (sections 3-4 indisponibles)')
print('Modèle agent :', settings.AGENT_MODEL_ID)

## 2. Test unitaire des tools « base de données » (sans modèle)

Ces tools interrogent directement l'ORM Django : ce sont les sources de vérité de l'agent.

In [ ]:
from assistant.agent import tools

print(tools.rechercher_cours('marketing'))

In [ ]:
print(tools.consulter_emploi_du_temps('L3', 'lundi'))

In [ ]:
print(tools.consulter_examens('L3'))

In [ ]:
# Lecture réelle du PDF déposé par l'administration (extrait)
print(tools.lire_document_cours('MKT301')[:800])

## 3. Test des tools « génération » (modèle Hugging Face requis)

`resumer_lecon` et `generer_quiz` lisent le PDF du cours puis appellent le modèle de génération.

In [ ]:
print(tools.resumer_lecon('CPT301', 'L3'))

In [ ]:
print(tools.generer_quiz('MKT301', 'L3', 3))

## 4. Test de l'agent complet (orchestration smolagents)

L'agent reçoit une question en langage naturel, **choisit lui-même ses tools** et compose la réponse.
On vérifie sur plusieurs intentions : planning, examen, pédagogie, conseil.

In [ ]:
from accounts.models import User
from assistant.agent import engine

amine = User.objects.get(username='amine')  # étudiant L3 de démo

def poser(question):
    reponse, tools_utilises = engine.repondre(question, amine)
    print('QUESTION :', question)
    print('TOOLS CHOISIS PAR L\'AGENT :', tools_utilises or '(aucun)')
    print('-' * 60)
    print(reponse)
    return reponse, tools_utilises

In [ ]:
_ = poser("Quel est mon emploi du temps de lundi ?")

In [ ]:
_ = poser("C'est quand mon prochain examen de comptabilité et qu'est-ce que j'ai le droit d'apporter ?")

In [ ]:
_ = poser("Je n'ai pas compris la différence entre segmentation et ciblage, tu peux m'expliquer simplement ?")

In [ ]:
_ = poser("Je suis débordé entre les partiels et mon projet de groupe, comment organiser ma semaine de révisions ?")

## 5. Bilan attendu

| Question | Besoin identifié | Tools attendus |
|---|---|---|
| emploi du temps de lundi | planning | `consulter_emploi_du_temps` |
| prochain examen de compta | examen | `consulter_examens` (et/ou `rechercher_cours`) |
| segmentation vs ciblage | pédagogie | `lire_document_cours` / `rechercher_cours` |
| organiser ma semaine | conseil | aucun (réponse directe du conseiller) |

L'agent répond en français, en markdown structuré, avec un ton adapté à un étudiant de L3 — et ne cite que des salles, horaires et consignes réellement présents dans la base de l'école.